# Uber Pickups — Détection des zones chaudes à New York

Uber veut aider ses chauffeurs à se positionner aux bons endroits, au bon moment. L'objectif ici est d'identifier les **hot-zones** (zones à forte demande) dans New York City à partir des données de pickups 2014.

**Plan :**
1. Chargement et préparation des données
2. Analyse exploratoire (volumes par heure et par jour)
3. Clustering avec KMeans — approche progressive
4. Clustering avec DBSCAN — alternative basée sur la densité
5. Comparaison des deux approches

## 1. Setup

In [1]:
import glob
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 2. Chargement des données

On charge les 6 fichiers 2014 (avril à septembre) qui contiennent les colonnes `Date/Time`, `Lat`, `Lon` et `Base`. On extrait ensuite les features temporelles utiles pour l'analyse.

In [2]:
# Chargement de tous les fichiers 2014 (lat/lon disponibles)
files = sorted(glob.glob('../data/raw/uber-raw-data-*14.csv'))

dfs = []
for f in files:
    tmp = pd.read_csv(f)
    tmp.columns = ['datetime', 'lat', 'lon', 'base']
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
df['datetime'] = pd.to_datetime(df['datetime'])

# Features temporelles
df['hour']      = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek   # 0 = lundi, 6 = dimanche
df['dayname']   = df['datetime'].dt.day_name()
df['month']     = df['datetime'].dt.month

print(f'Total pickups : {len(df):,}')
print(f'Période       : {df["datetime"].min().date()} -> {df["datetime"].max().date()}')
df.head()

Total pickups : 4,534,327
Période       : 2014-04-01 -> 2014-09-30


,datetime,lat,lon,base,hour,dayofweek,dayname,month
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,0,1,Tuesday,4
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,0,1,Tuesday,4
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,0,1,Tuesday,4
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,0,1,Tuesday,4
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,0,1,Tuesday,4


## 3. Analyse exploratoire

Avant de lancer le clustering, on regarde comment se répartissent les pickups dans le temps pour identifier les pics de demande.

### 3.1 Distribution horaire

In [3]:
by_hour = df.groupby('hour').size().reset_index(name='pickups')

fig = px.bar(by_hour, x='hour', y='pickups',
             title='Nombre de pickups par heure de la journée',
             labels={'hour': 'Heure', 'pickups': 'Nombre de pickups'},
             color='pickups', color_continuous_scale='Blues')
fig.show()

### 3.2 Distribution par jour de la semaine

In [4]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_fr    = {'Monday': 'Lundi', 'Tuesday': 'Mardi', 'Wednesday': 'Mercredi',
             'Thursday': 'Jeudi', 'Friday': 'Vendredi', 'Saturday': 'Samedi', 'Sunday': 'Dimanche'}

by_day = df.groupby('dayname').size().reset_index(name='pickups')
by_day['jour']  = by_day['dayname'].map(day_fr)
by_day['order'] = by_day['dayname'].map({d: i for i, d in enumerate(day_order)})
by_day = by_day.sort_values('order')

fig = px.bar(by_day, x='jour', y='pickups',
             title='Nombre de pickups par jour de la semaine',
             labels={'jour': 'Jour', 'pickups': 'Nombre de pickups'},
             color='pickups', color_continuous_scale='Purples')
fig.show()

### 3.3 Heatmap jour × heure

Cette vue croisée permet d'identifier les combinaisons jour/heure les plus chargées d'un seul coup d'œil.

In [5]:
heat = df.groupby(['dayofweek', 'hour']).size().reset_index(name='pickups')
heat_piv = heat.pivot(index='dayofweek', columns='hour', values='pickups')
heat_piv.index = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']

fig = px.imshow(heat_piv,
                title='Heatmap — pickups par jour et heure',
                labels=dict(x='Heure', y='Jour', color='Pickups'),
                color_continuous_scale='YlOrRd',
                aspect='auto')
fig.show()

**Observations :**
- Le pic de demande se situe en fin d'après-midi / soirée (17h–21h)
- Le vendredi et le samedi soir sont les moments les plus chargés
- La nuit (1h–6h) reste plus calme sauf le week-end

Ces informations guident le choix du créneau pour commencer le clustering.

## 4. Clustering — KMeans

### 4.1 Commencer petit : vendredi à 18h

On applique d'abord KMeans sur un seul créneau (vendredi 18h, l'un des pics identifiés) pour valider l'approche avant de généraliser.

In [6]:
fri_6pm = df[(df['dayofweek'] == 4) & (df['hour'] == 18)].copy()
print(f'Pickups vendredi 18h : {len(fri_6pm):,}')

km_small = KMeans(n_clusters=8, random_state=42, n_init=10)
fri_6pm['cluster'] = km_small.fit_predict(fri_6pm[['lat', 'lon']].values).astype(str)

fig = px.scatter_mapbox(
    fri_6pm.sample(min(5000, len(fri_6pm)), random_state=42),
    lat='lat', lon='lon', color='cluster',
    mapbox_style='open-street-map', zoom=10,
    center={'lat': 40.73, 'lon': -73.99},
    title='KMeans (k=8) — Vendredi 18h',
    opacity=0.6
)
fig.update_traces(marker_size=5)
fig.show()

Pickups vendredi 18h : 54,762


### 4.2 Généralisation par jour de la semaine

On applique KMeans sur l'ensemble des pickups de chaque jour de la semaine et on affiche les **centres des clusters** sur la carte. Ces centres représentent directement les zones chaudes à recommander aux chauffeurs.

In [7]:
day_order_en = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_order_fr = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']

centers_list = []
for i, day in enumerate(day_order_en):
    data = df[df['dayname'] == day][['lat', 'lon']].dropna()
    km = KMeans(n_clusters=8, random_state=42, n_init=10)
    km.fit(data)
    c = pd.DataFrame(km.cluster_centers_, columns=['lat', 'lon'])
    c['jour'] = day_order_fr[i]
    centers_list.append(c)

centers_df = pd.concat(centers_list, ignore_index=True)
centers_df['taille'] = 1

fig = px.scatter_mapbox(
    centers_df, lat='lat', lon='lon',
    color='jour', size='taille', size_max=18,
    mapbox_style='open-street-map', zoom=10,
    center={'lat': 40.73, 'lon': -73.99},
    title='Zones chaudes par jour de la semaine — Centres KMeans (k=8)'
)
fig.show()

## 5. Clustering — DBSCAN

DBSCAN (Density-Based Spatial Clustering) identifie les zones denses sans qu'on ait à fixer le nombre de clusters à l'avance. Les points isolés sont marqués comme **bruit** (cluster = -1).

On travaille sur un échantillon de 50 000 points pour garder des temps de calcul raisonnables.

In [8]:
df_db = df.sample(50000, random_state=42).copy()

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(df_db[['lat', 'lon']].values)

db = DBSCAN(eps=0.2, min_samples=100)
df_db['cluster'] = db.fit_predict(coords_scaled)

n_clusters = len(set(df_db['cluster'])) - (1 if -1 in df_db['cluster'].values else 0)
n_noise    = (df_db['cluster'] == -1).sum()
print(f'Clusters détectés : {n_clusters}')
print(f'Points bruit      : {n_noise:,} ({n_noise / len(df_db) * 100:.1f}%)')

df_db_clean = df_db[df_db['cluster'] != -1].copy()
df_db_clean['cluster'] = df_db_clean['cluster'].astype(str)

fig = px.scatter_mapbox(
    df_db_clean,
    lat='lat', lon='lon', color='cluster',
    mapbox_style='open-street-map', zoom=10,
    center={'lat': 40.73, 'lon': -73.99},
    title=f'DBSCAN — {n_clusters} clusters identifiés (points bruit exclus)',
    opacity=0.6
)
fig.update_traces(marker_size=5)
fig.show()

Clusters détectés : 6
Points bruit      : 2,481 (5.0%)


## 6. Comparaison KMeans vs DBSCAN

In [9]:
comparison = pd.DataFrame({
    'Critère': [
        'Nombre de clusters',
        'Points non assignés',
        'Forme des clusters',
        'Paramètre clé',
        'Vitesse'
    ],
    'KMeans': [
        'Fixé (k=8)',
        'Aucun (tous assignés)',
        'Sphérique / régulière',
        'Nombre de clusters k',
        'Très rapide'
    ],
    'DBSCAN': [
        str(n_clusters),
        f'{n_noise:,} points bruit',
        'Variable (densité)',
        'eps, min_samples',
        'Plus lent sur grands datasets'
    ]
})
comparison

,Critère,KMeans,DBSCAN
0,Nombre de clusters,Fixé (k=8),6
1,Points non assignés,Aucun (tous assignés),"2,481 points bruit"
2,Forme des clusters,Sphérique / régulière,Variable (densité)
3,Paramètre clé,Nombre de clusters k,"eps, min_samples"
4,Vitesse,Très rapide,Plus lent sur grands datasets


## Conclusion

Les deux algorithmes donnent des résultats cohérents et complémentaires :

- **KMeans** est simple, rapide et facile à interpréter. En fixant k=8 zones par jour, on obtient une carte claire des hot-zones à recommander aux chauffeurs. Les centres de clusters se concentrent logiquement autour de Midtown Manhattan, Lower Manhattan, les aéroports (JFK, LaGuardia) et les grandes gares.

- **DBSCAN** détecte automatiquement les zones denses sans imposer un nombre de clusters. Il identifie des formes plus complexes mais génère du bruit (points isolés non assignés). Utile pour valider les résultats de KMeans.

**Recommandation :** Pour une intégration dans l'application, KMeans par créneau horaire et jour de la semaine est l'approche la plus directement exploitable. DBSCAN peut servir en complément pour détecter des zones émergentes.